In [0]:
%sql
CREATE OR REPLACE TABLE dev.sales.zorder_perf_drill
USING DELTA
AS
SELECT /*+ REPARTITION(40) */
    CAST(id AS BIGINT) AS order_id,
    CAST(id % 1000 AS INT) AS customer_id,
    CAST(id % 100 AS INT) AS amount,
    CASE 
        WHEN id % 3 = 0 THEN 'PAID'
        WHEN id % 3 = 1 THEN 'PENDING'
        ELSE 'FAILED'
    END AS status,
    date_add(DATE '2026-01-01', CAST(id % 30 AS INT)) AS order_date
FROM range(200000);

In [0]:
%sql
DESCRIBE DETAIL dev.sales.zorder_perf_drill;

In [0]:
%sql
SELECT 
    customer_id,
    COUNT(*) AS orders_count,
    SUM(amount) AS total_amount
FROM dev.sales.zorder_perf_drill
WHERE customer_id = 42
GROUP BY customer_id;

In [0]:
%sql
OPTIMIZE dev.sales.zorder_perf_drill
ZORDER BY (customer_id);

In [0]:
%sql
DESCRIBE HISTORY dev.sales.zorder_perf_drill;

In [0]:
%sql
SELECT 
    customer_id,
    COUNT(*) AS orders_count,
    SUM(amount) AS total_amount
FROM dev.sales.zorder_perf_drill
WHERE customer_id = 42
GROUP BY customer_id;

In [0]:
%sql
DESCRIBE DETAIL dev.sales.zorder_perf_drill;

In [0]:
# before_zorder_files_scanned = 40
# before_zorder_bytes_read = 122396
# before_zorder_runtime = 0.977s runtime

# after_zorder_files_scanned = 1
# after_zorder_bytes_read = 26790
# after_zorder_runtime = 0.887s runtime 